# 14r CLIP-ReID Stage-2 Recovery (VeRi-776)

Spec: `docs/subagent-specs/14r-recovery.md`, locked plan dated 2026-05-10.

This kernel loads the completed Stage 1 prompt checkpoint from `gumfreddy/14r-clip-reid-stage1-prompts` and runs Stage 2 only: 60 epochs, P=16/K=4 batch 64, sqrt-scaled learning rates, Stage-2-only walltime guard, and the same four-row final evaluation suite.

In [ ]:
import json
import os
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


pip_install("timm>=1.0,<2.0", "torch>=2.4", "torchvision>=0.19", "scikit-learn", "matplotlib")

import timm
import torch
import torchvision

print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}, indent=2))

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working")
RUN_DIR = OUTPUT_DIR / "14r_recovery_stage2"
RUN_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_PATH = Path("/kaggle/input/14r-clip-reid-prompts-public/clip_reid_vit_b16_veri776_stage1_prompts.pt")
BEST_MAP_PATH = OUTPUT_DIR / "clip_reid_vit_b16_veri776_stage2_best_mAP.pth"
BEST_R1_PATH = OUTPUT_DIR / "clip_reid_vit_b16_veri776_stage2_best_R1.pth"
LAST_PATH = OUTPUT_DIR / "clip_reid_vit_b16_veri776_stage2_last.pth"
TRAIN_LOG_PATH = OUTPUT_DIR / "train_log.json"
EVAL_RESULTS_PATH = OUTPUT_DIR / "eval_results.json"
RECIPE_PATH = OUTPUT_DIR / "recipe.json"
SUMMARY_PATH = OUTPUT_DIR / "14r_recovery_summary.json"

TIMM_MODEL_NAME = "vit_base_patch16_clip_224.openai"
IMG_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = 196
FEATURE_DIM = 768
TEXT_DIM = 512
CONCAT_FEATURE_DIM = 1536
NUM_VERI_CAMERAS = 20
EXPECTED_NUM_CLASSES = 576

STAGE2_EPOCHS = 60
WARMUP_EPOCHS = 5
BACKBONE_LR = 0.000495
HEAD_LR = 0.00495
MIN_LR = 1e-6
WARMUP_START_LR = 1e-7
WEIGHT_DECAY = 1e-4
TRIPLET_MARGIN = 0.3
LABEL_SMOOTHING = 0.1
I2T_TEMPERATURE = 0.07
LAMBDA_GLOBAL_CE = 1.0
LAMBDA_GLOBAL_TRI = 1.0
LAMBDA_I2TCE = 1.0
LAMBDA_JPM_CE = 1.0

P_IDS = 16
K_INSTANCES = 4
BATCH_SIZE = P_IDS * K_INSTANCES
ACCUM_STEPS = 1
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 4
PERIODIC_EVAL_EPOCHS = {20, 40, 50, 55, 60}
MAX_STAGE2_TRAIN_HOURS = 7.0
EPOCH1_MAX_MINUTES = 8.0

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
ERASE_VALUE = [0.4914, 0.4822, 0.4465]


def find_veri_root():
    candidates = [
        Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
        Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    ]
    for candidate in candidates:
        roots = [candidate] + list(candidate.glob("**/*")) if candidate.exists() else []
        for root in roots:
            if (root / "image_train").is_dir() and (root / "image_query").is_dir() and (root / "image_test").is_dir():
                return root
    for root, dirs, files in os.walk("/kaggle/input"):
        path = Path(root)
        if (path / "image_train").is_dir() and (path / "image_query").is_dir() and (path / "image_test").is_dir():
            return path
    raise RuntimeError("VeRi-776 not found. Attach abhyudaya12/veri-vehicle-re-identification-dataset.")


VERI_ROOT = find_veri_root()
print("VeRi root:", VERI_ROOT)
print("Device:", DEVICE)

In [ ]:
CAMERA_PATTERN = re.compile(r"^(?P<pid>-?\d+)_c(?P<camid>\d+)")


def parse_split(split_dir, relabel_train=False):
    items = []
    raw_pids = []
    for image_path in sorted(Path(split_dir).glob("*.jpg")):
        match = CAMERA_PATTERN.match(image_path.name)
        if match is None:
            continue
        pid = int(match.group("pid"))
        if pid == -1:
            continue
        camid = int(match.group("camid"))
        if not 1 <= camid <= NUM_VERI_CAMERAS:
            raise RuntimeError(f"Camera ID out of range in {image_path.name}: {camid}")
        items.append({"path": str(image_path), "pid": pid, "camid": camid - 1})
        raw_pids.append(pid)
    if relabel_train:
        pid_to_label = {pid: index for index, pid in enumerate(sorted(set(raw_pids)))}
        for item in items:
            item["pid"] = pid_to_label[item["pid"]]
    return items


train_items = parse_split(VERI_ROOT / "image_train", relabel_train=True)
query_items = parse_split(VERI_ROOT / "image_query", relabel_train=False)
gallery_items = parse_split(VERI_ROOT / "image_test", relabel_train=False)
NUM_CLASSES = len({item["pid"] for item in train_items})
assert NUM_CLASSES == EXPECTED_NUM_CLASSES, f"Expected 576 train IDs, got {NUM_CLASSES}"

resolved_cfg = timm.data.resolve_model_data_config(timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMG_SIZE))
assert tuple(round(value, 8) for value in resolved_cfg["mean"]) == tuple(round(value, 8) for value in CLIP_MEAN)
assert tuple(round(value, 8) for value in resolved_cfg["std"]) == tuple(round(value, 8) for value in CLIP_STD)

stage2_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.4), ratio=(0.3, 3.3333333333), value=ERASE_VALUE),
])

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


class VeRiDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        image = Image.open(item["path"]).convert("RGB")
        return self.transform(image), item["pid"], item["camid"], item["path"]


class RandomIdentitySampler(Sampler):
    def __init__(self, items, num_pids_per_batch, num_instances):
        self.items = items
        self.num_pids_per_batch = num_pids_per_batch
        self.num_instances = num_instances
        self.index_dic = defaultdict(list)
        for index, item in enumerate(items):
            self.index_dic[item["pid"]].append(index)
        self.pids = sorted(self.index_dic.keys())
        self.length = 0
        for pid in self.pids:
            count = len(self.index_dic[pid])
            self.length += max(count, self.num_instances) - max(count, self.num_instances) % self.num_instances

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)
        for pid in self.pids:
            idxs = self.index_dic[pid].copy()
            if len(idxs) < self.num_instances:
                idxs = np.random.choice(idxs, size=self.num_instances, replace=True).tolist()
            random.shuffle(idxs)
            batch_idxs = []
            for idx in idxs:
                batch_idxs.append(idx)
                if len(batch_idxs) == self.num_instances:
                    batch_idxs_dict[pid].append(batch_idxs)
                    batch_idxs = []
        avai_pids = self.pids.copy()
        final_idxs = []
        while len(avai_pids) >= self.num_pids_per_batch:
            selected_pids = random.sample(avai_pids, self.num_pids_per_batch)
            for pid in selected_pids:
                final_idxs.extend(batch_idxs_dict[pid].pop(0))
                if len(batch_idxs_dict[pid]) == 0:
                    avai_pids.remove(pid)
        return iter(final_idxs)

    def __len__(self):
        return self.length


stage2_train_loader = DataLoader(
    VeRiDataset(train_items, stage2_transform),
    batch_size=BATCH_SIZE,
    sampler=RandomIdentitySampler(train_items, P_IDS, K_INSTANCES),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(VeRiDataset(query_items, test_transform), batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
gallery_loader = DataLoader(VeRiDataset(gallery_items, test_transform), batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(json.dumps({"train": len(train_items), "query": len(query_items), "gallery": len(gallery_items), "classes": NUM_CLASSES, "batch_size": BATCH_SIZE}, indent=2))

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.log_softmax(inputs)
        one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        smoothed = (1 - self.epsilon) * one_hot + self.epsilon / self.num_classes
        return (-smoothed * log_probs).sum(dim=1).mean()


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, features, labels):
        distance = torch.cdist(features.float(), features.float(), p=2)
        labels = labels.view(-1, 1)
        positive_mask = labels.eq(labels.t())
        negative_mask = ~positive_mask
        dist_ap = (distance * positive_mask.float()).max(dim=1)[0]
        dist_an = distance.masked_fill(~negative_mask, 1e9).min(dim=1)[0]
        return F.relu(dist_ap - dist_an + self.margin).mean()


class TransReIDClipReID(nn.Module):
    def __init__(self, num_classes, num_cameras=20, pretrained=True):
        super().__init__()
        self.vit = timm.create_model(TIMM_MODEL_NAME, pretrained=pretrained, num_classes=0, img_size=IMG_SIZE)
        assert self.vit.patch_embed.num_patches == NUM_PATCHES
        assert self.vit.embed_dim == FEATURE_DIM
        self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, FEATURE_DIM))
        nn.init.trunc_normal_(self.sie_embed, std=0.02)
        self.bn = nn.BatchNorm1d(FEATURE_DIM)
        self.bn.bias.requires_grad_(False)
        self.classifier = nn.Linear(FEATURE_DIM, num_classes, bias=False)
        self.jpm_bns = nn.ModuleList([nn.BatchNorm1d(FEATURE_DIM) for _ in range(4)])
        for bn in self.jpm_bns:
            bn.bias.requires_grad_(False)
        self.jpm_classifiers = nn.ModuleList([nn.Linear(FEATURE_DIM, num_classes, bias=False) for _ in range(4)])
        self.i2t_projection = nn.Linear(FEATURE_DIM, TEXT_DIM, bias=False)
        nn.init.xavier_uniform_(self.i2t_projection.weight)

    def forward_features_tokens(self, images, camids):
        x = self.vit.patch_embed(images)
        x = self.vit._pos_embed(x)
        x = x + self.sie_embed[camids]
        x = self.vit.patch_drop(x)
        x = self.vit.norm_pre(x)
        x = self.vit.blocks(x)
        x = self.vit.norm(x)
        return x

    def forward(self, images, camids):
        tokens = self.forward_features_tokens(images, camids)
        global_feat = tokens[:, 0]
        bn_feat = self.bn(global_feat)
        logits = self.classifier(bn_feat)
        patch_tokens = tokens[:, 1:]
        groups = torch.chunk(patch_tokens, 4, dim=1)
        jpm_feats = [group.mean(dim=1) for group in groups]
        jpm_bn_feats = [bn(feat) for bn, feat in zip(self.jpm_bns, jpm_feats)]
        jpm_logits = [classifier(feat) for classifier, feat in zip(self.jpm_classifiers, jpm_bn_feats)]
        projected_i2t = self.i2t_projection(global_feat)
        return {"tokens": tokens, "global_feat": global_feat, "bn_feat": bn_feat, "logits": logits, "jpm_logits": jpm_logits, "i2t_feat": projected_i2t}


LLRD_FACTOR = 0.65


def build_optimizer(model):
    head_params = []
    backbone_groups = defaultdict(list)
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("vit."):
            depth = 0
            match = re.search(r"vit\.blocks\.(\d+)", name)
            if match:
                depth = int(match.group(1)) + 1
            elif "patch_embed" in name or "pos_embed" in name or "cls_token" in name:
                depth = 0
            else:
                depth = 12
            lr = BACKBONE_LR * (LLRD_FACTOR ** (12 - depth))
            backbone_groups[lr].append(parameter)
        else:
            head_params.append(parameter)
    groups = [{"params": params, "lr": lr, "name": f"backbone_lr_{lr:.8g}"} for lr, params in sorted(backbone_groups.items())]
    groups.append({"params": head_params, "lr": HEAD_LR, "name": "heads"})
    return torch.optim.AdamW(groups, lr=BACKBONE_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))


def set_epoch_lr(optimizer, epoch, total_epochs):
    if epoch <= WARMUP_EPOCHS:
        progress = epoch / max(1, WARMUP_EPOCHS)
        factor = (WARMUP_START_LR / BACKBONE_LR) + progress * (1 - WARMUP_START_LR / BACKBONE_LR)
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(1, total_epochs - WARMUP_EPOCHS)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        factor = (MIN_LR / BACKBONE_LR) + (1 - MIN_LR / BACKBONE_LR) * cosine
    for group in optimizer.param_groups:
        if group.get("name") == "heads":
            group["lr"] = HEAD_LR * factor
        else:
            base_lr = float(group["name"].replace("backbone_lr_", ""))
            group["lr"] = base_lr * factor


ce_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTHING)
triplet_loss_fn = TripletLoss(TRIPLET_MARGIN)
model = TransReIDClipReID(NUM_CLASSES, NUM_VERI_CAMERAS, pretrained=True).to(DEVICE)
optimizer = build_optimizer(model)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

print(json.dumps({
    "model": TIMM_MODEL_NAME,
    "params_m": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "optimizer_groups": [{"name": group.get("name"), "lr": group["lr"], "params": sum(p.numel() for p in group["params"])} for group in optimizer.param_groups],
}, indent=2))

In [ ]:
def to_metric_dict(mAP, cmc):
    return {"mAP": float(mAP), "R1": float(cmc[0]), "R5": float(cmc[4]), "R10": float(cmc[9])}


def evaluate_rank(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=50):
    num_q, num_g = distmat.shape
    if num_g < max_rank:
        max_rank = num_g
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc = []
    all_ap = []
    for q_idx in range(num_q):
        order = indices[q_idx]
        remove = (g_pids[order] == q_pids[q_idx]) & (g_camids[order] == q_camids[q_idx])
        keep = np.invert(remove)
        raw_cmc = matches[q_idx][keep]
        if not np.any(raw_cmc):
            continue
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        num_rel = raw_cmc.sum()
        tmp_cmc = raw_cmc.cumsum()
        precision = tmp_cmc / (np.arange(len(raw_cmc)) + 1)
        all_ap.append((precision * raw_cmc).sum() / num_rel)
    if not all_cmc:
        raise RuntimeError("No valid query for evaluation")
    return np.asarray(all_cmc).astype(np.float32).sum(0) / len(all_cmc), float(np.mean(all_ap))


@torch.no_grad()
def extract_features(model, loader, flip=False, concat_patch=False):
    model.eval()
    features, pids, camids = [], [], []
    for images, batch_pids, batch_camids, paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        batch_camids = batch_camids.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(images, batch_camids)
            cls_feat = outputs["bn_feat"]
            patch_feat = outputs["tokens"][:, 1:].mean(dim=1)
            if flip:
                flipped = torch.flip(images, dims=[3])
                flip_outputs = model(flipped, batch_camids)
                cls_feat = 0.5 * (cls_feat + flip_outputs["bn_feat"])
                patch_feat = 0.5 * (patch_feat + flip_outputs["tokens"][:, 1:].mean(dim=1))
            feat = torch.cat([cls_feat, patch_feat], dim=1) if concat_patch else cls_feat
            feat = F.normalize(feat.float(), dim=1)
        features.append(feat.cpu().numpy())
        pids.extend(batch_pids.numpy().tolist())
        camids.extend(batch_camids.cpu().numpy().tolist())
    return np.concatenate(features, axis=0), np.asarray(pids), np.asarray(camids)


def average_query_expansion(features, k=2, iterations=1):
    features = features.astype(np.float32)
    for _ in range(iterations):
        sims = features @ features.T
        topk = np.argsort(-sims, axis=1)[:, :k]
        features = np.stack([features[indexes].mean(axis=0) for indexes in topk])
        features /= np.linalg.norm(features, axis=1, keepdims=True) + 1e-12
    return features


def re_ranking(q_g_dist, q_q_dist, g_g_dist, k1=80, k2=15, lambda_value=0.2):
    original_dist = np.concatenate(
        [np.concatenate([q_q_dist, q_g_dist], axis=1), np.concatenate([q_g_dist.T, g_g_dist], axis=1)],
        axis=0,
    )
    original_dist = np.power(original_dist, 2).astype(np.float32)
    original_dist = np.transpose(1.0 * original_dist / np.max(original_dist, axis=0))
    all_num = original_dist.shape[0]
    gallery_num = q_g_dist.shape[1]
    V = np.zeros_like(original_dist, dtype=np.float32)
    initial_rank = np.argsort(original_dist).astype(np.int32)
    for i in range(all_num):
        forward_k_neigh_index = initial_rank[i, : k1 + 1]
        backward_k_neigh_index = initial_rank[forward_k_neigh_index, : k1 + 1]
        fi = np.where(backward_k_neigh_index == i)[0]
        k_reciprocal_index = forward_k_neigh_index[fi]
        k_reciprocal_expansion_index = k_reciprocal_index
        for candidate in k_reciprocal_index:
            candidate_forward = initial_rank[candidate, : int(np.around(k1 / 2)) + 1]
            candidate_backward = initial_rank[candidate_forward, : int(np.around(k1 / 2)) + 1]
            fi_candidate = np.where(candidate_backward == candidate)[0]
            candidate_reciprocal = candidate_forward[fi_candidate]
            if len(np.intersect1d(candidate_reciprocal, k_reciprocal_index)) > 2 / 3 * len(candidate_reciprocal):
                k_reciprocal_expansion_index = np.append(k_reciprocal_expansion_index, candidate_reciprocal)
        k_reciprocal_expansion_index = np.unique(k_reciprocal_expansion_index)
        weight = np.exp(-original_dist[i, k_reciprocal_expansion_index])
        V[i, k_reciprocal_expansion_index] = 1.0 * weight / np.sum(weight)
    if k2 != 1:
        V_qe = np.zeros_like(V, dtype=np.float32)
        for i in range(all_num):
            V_qe[i, :] = np.mean(V[initial_rank[i, :k2], :], axis=0)
        V = V_qe
    inv_index = [np.where(V[:, i] != 0)[0] for i in range(all_num)]
    jaccard_dist = np.zeros_like(original_dist, dtype=np.float32)
    for i in range(all_num):
        temp_min = np.zeros((1, all_num), dtype=np.float32)
        ind_non_zero = np.where(V[i, :] != 0)[0]
        ind_images = [inv_index[ind] for ind in ind_non_zero]
        for j, ind in enumerate(ind_non_zero):
            temp_min[0, ind_images[j]] += np.minimum(V[i, ind], V[ind_images[j], ind])
        jaccard_dist[i] = 1 - temp_min / (2 - temp_min)
    final_dist = jaccard_dist[: q_g_dist.shape[0], q_g_dist.shape[0] :]
    final_dist = final_dist * (1 - lambda_value) + q_g_dist * lambda_value
    return final_dist[:, :gallery_num]


def evaluate_feature_row(row_name, concat_patch=False, aqe_k=None, rerank=False):
    qf, q_pids, q_camids = extract_features(model, query_loader, flip=True, concat_patch=concat_patch)
    gf, g_pids, g_camids = extract_features(model, gallery_loader, flip=True, concat_patch=concat_patch)
    if aqe_k is not None:
        combined = average_query_expansion(np.vstack([qf, gf]), k=aqe_k, iterations=1)
        qf = combined[: len(qf)]
        gf = combined[len(qf) :]
    q_g_dist = 1 - np.clip(qf @ gf.T, -1, 1)
    if rerank:
        q_q_dist = 1 - np.clip(qf @ qf.T, -1, 1)
        g_g_dist = 1 - np.clip(gf @ gf.T, -1, 1)
        q_g_dist = re_ranking(q_g_dist, q_q_dist, g_g_dist, k1=80, k2=15, lambda_value=0.2)
    cmc, mAP = evaluate_rank(q_g_dist, q_pids, g_pids, q_camids, g_camids)
    result = {"row": row_name, **to_metric_dict(mAP, cmc), "concat_patch": concat_patch, "aqe_k": aqe_k, "rerank": rerank}
    print(json.dumps(result, indent=2))
    return result

In [ ]:
recipe = {
    "experiment": "14r recovery CLIP-ReID Stage-2-only VeRi-776",
    "spec": "docs/subagent-specs/14r-recovery.md#LOCKED PLAN - 2026-05-10",
    "backbone": TIMM_MODEL_NAME,
    "source_prompts": {
        "dataset": "gumfreddy/14r-clip-reid-stage1-prompts",
        "path": str(PROMPTS_PATH),
        "expected_text_features_shape": [EXPECTED_NUM_CLASSES, TEXT_DIM],
        "expected_ctx_shape": [4, TEXT_DIM],
        "expected_id_tokens_shape": [EXPECTED_NUM_CLASSES, TEXT_DIM],
    },
    "image_size": IMG_SIZE,
    "stage2": {
        "epochs": STAGE2_EPOCHS,
        "optimizer": "AdamW",
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "llrd_factor": LLRD_FACTOR,
        "warmup_epochs": WARMUP_EPOCHS,
        "min_lr": MIN_LR,
        "batch": {"P": P_IDS, "K": K_INSTANCES, "size": BATCH_SIZE},
        "losses": {"ce": LAMBDA_GLOBAL_CE, "triplet": LAMBDA_GLOBAL_TRI, "i2tce": LAMBDA_I2TCE, "jpm_ce": LAMBDA_JPM_CE},
        "augmentation": ["resize_bicubic_224", "horizontal_flip_0.5", "pad_10", "random_crop_224", "random_erasing_0.5"],
        "periodic_eval_epochs": sorted(PERIODIC_EVAL_EPOCHS),
        "max_stage2_train_hours": MAX_STAGE2_TRAIN_HOURS,
    },
    "eval_suite": ["single_flip_cls_base", "single_flip_cls_aqe2_rerank", "concat_patch_flip_aqe2_rerank", "concat_patch_flip_aqe3_rerank"],
    "rerank": {"k1": 80, "k2": 15, "lambda": 0.2},
    "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
    "verdict_gate": "WIN if best concat AQE row mAP>=0.9154 and R1>=0.9833; MARGINAL if mAP>=0.905 or R1>=0.980; else FAIL",
}
RECIPE_PATH.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory free before Stage 2: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB / {torch.cuda.mem_get_info()[1] / 1024**3:.2f} GB")

In [ ]:
def flush_train_log():
    TRAIN_LOG_PATH.write_text(json.dumps(train_log, indent=2), encoding="utf-8")


def save_and_abort(message):
    flush_train_log()
    torch.save(model.state_dict(), LAST_PATH)
    raise RuntimeError(message)


prompt_payload = torch.load(PROMPTS_PATH, map_location="cpu")
required_prompt_keys = {"text_features", "ctx", "id_tokens", "recipe"}
missing_prompt_keys = sorted(required_prompt_keys - set(prompt_payload.keys()))
if missing_prompt_keys:
    raise RuntimeError(f"Prompt checkpoint missing keys: {missing_prompt_keys}")

text_features = prompt_payload["text_features"].float()
ctx = prompt_payload["ctx"].float()
id_tokens = prompt_payload["id_tokens"].float()
assert tuple(text_features.shape) == (EXPECTED_NUM_CLASSES, TEXT_DIM), tuple(text_features.shape)
assert tuple(ctx.shape) == (4, TEXT_DIM), tuple(ctx.shape)
assert tuple(id_tokens.shape) == (EXPECTED_NUM_CLASSES, TEXT_DIM), tuple(id_tokens.shape)
assert torch.isfinite(text_features).all(), "Non-finite text_features in prompt checkpoint"
assert torch.isfinite(ctx).all(), "Non-finite ctx in prompt checkpoint"
assert torch.isfinite(id_tokens).all(), "Non-finite id_tokens in prompt checkpoint"

cached_text_features = F.normalize(text_features.to(DEVICE), dim=-1)
train_log = {
    "stage2": [],
    "periodic_eval": [],
    "prompt_checkpoint": {
        "path": str(PROMPTS_PATH),
        "text_features_shape": list(text_features.shape),
        "ctx_shape": list(ctx.shape),
        "id_tokens_shape": list(id_tokens.shape),
        "text_feature_norm_mean": float(text_features.norm(dim=1).mean()),
    },
}
flush_train_log()
print(json.dumps(train_log["prompt_checkpoint"], indent=2))

best_map = -1.0
best_r1 = -1.0
stage2_start = time.time()

for epoch in range(1, STAGE2_EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    set_epoch_lr(optimizer, epoch, STAGE2_EPOCHS)
    epoch_losses = []
    for images, pids, camids, paths in stage2_train_loader:
        images = images.to(DEVICE, non_blocking=True)
        pids = pids.to(DEVICE, non_blocking=True)
        camids = camids.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(images, camids)
            loss_ce = ce_loss_fn(outputs["logits"].float(), pids)
            loss_tri = triplet_loss_fn(outputs["global_feat"].float(), pids)
            i2t_logits = F.normalize(outputs["i2t_feat"].float(), dim=-1) @ cached_text_features.t() / I2T_TEMPERATURE
            loss_i2t = F.cross_entropy(i2t_logits, pids)
            loss_jpm = sum(ce_loss_fn(logits.float(), pids) for logits in outputs["jpm_logits"]) / len(outputs["jpm_logits"])
            loss = LAMBDA_GLOBAL_CE * loss_ce + LAMBDA_GLOBAL_TRI * loss_tri + LAMBDA_I2TCE * loss_i2t + LAMBDA_JPM_CE * loss_jpm
        if not torch.isfinite(loss):
            row = {"epoch": epoch, "loss": float("nan"), "abort_reason": "non_finite_loss"}
            train_log["stage2"].append(row)
            save_and_abort(f"Non-finite Stage 2 loss at epoch {epoch}")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(float(loss.detach().cpu()))
    elapsed_hours = (time.time() - stage2_start) / 3600.0
    projected_total = elapsed_hours / epoch * STAGE2_EPOCHS
    epoch_minutes = (time.time() - epoch_start) / 60.0
    mean_loss = float(np.mean(epoch_losses))
    row = {
        "epoch": epoch,
        "loss": mean_loss,
        "elapsed_hours": elapsed_hours,
        "projected_stage2_hours": projected_total,
        "epoch_minutes": epoch_minutes,
        "lr_head": optimizer.param_groups[-1]["lr"],
    }
    train_log["stage2"].append(row)
    print(json.dumps({"stage": 2, **row}))
    flush_train_log()
    torch.save(model.state_dict(), LAST_PATH)

    if epoch == 1 and epoch_minutes > EPOCH1_MAX_MINUTES:
        save_and_abort(f"Stage 2 epoch 1 elapsed {epoch_minutes:.2f} min exceeds {EPOCH1_MAX_MINUTES:.2f} min fast-fail guard")
    if epoch == 1 and mean_loss > 12.0:
        save_and_abort(f"Stage 2 epoch 1 loss {mean_loss:.4f} exceeds 12.0 fast-fail guard")
    if projected_total > MAX_STAGE2_TRAIN_HOURS:
        if epoch < 20:
            save_and_abort(
                f"Projected Stage-2 walltime {projected_total:.2f}h exceeds {MAX_STAGE2_TRAIN_HOURS}h guard before epoch 20; saved log and last checkpoint"
            )
        save_and_abort(
            f"Projected Stage-2 walltime {projected_total:.2f}h exceeds {MAX_STAGE2_TRAIN_HOURS}h guard; saved log and last checkpoint at epoch {epoch}"
        )
    if epoch in PERIODIC_EVAL_EPOCHS:
        result = evaluate_feature_row(f"epoch_{epoch}_single_flip_cls_base", concat_patch=False, aqe_k=None, rerank=False)
        train_log["periodic_eval"].append({"epoch": epoch, **result})
        if result["mAP"] > best_map:
            best_map = result["mAP"]
            torch.save(model.state_dict(), BEST_MAP_PATH)
        if result["R1"] > best_r1:
            best_r1 = result["R1"]
            torch.save(model.state_dict(), BEST_R1_PATH)
        flush_train_log()
        torch.save(model.state_dict(), LAST_PATH)
        if epoch == 20 and result["mAP"] < 0.55:
            save_and_abort(f"Epoch 20 base mAP {result['mAP']:.4f} below 0.55 fast-fail guard")

if not BEST_MAP_PATH.exists():
    torch.save(model.state_dict(), BEST_MAP_PATH)
if not BEST_R1_PATH.exists():
    torch.save(model.state_dict(), BEST_R1_PATH)
flush_train_log()
print(json.dumps({"stage2_done": True, "last": str(LAST_PATH), "best_mAP": str(BEST_MAP_PATH), "best_R1": str(BEST_R1_PATH)}, indent=2))

In [ ]:
model.load_state_dict(torch.load(BEST_MAP_PATH, map_location=DEVICE))
eval_rows = [
    evaluate_feature_row("single_flip_cls_base", concat_patch=False, aqe_k=None, rerank=False),
    evaluate_feature_row("single_flip_cls_aqe2_rerank_k1_80_k2_15_lambda_0_2", concat_patch=False, aqe_k=2, rerank=True),
    evaluate_feature_row("concat_patch_flip_aqe2_rerank_k1_80_k2_15_lambda_0_2", concat_patch=True, aqe_k=2, rerank=True),
    evaluate_feature_row("concat_patch_flip_aqe3_rerank_k1_80_k2_15_lambda_0_2", concat_patch=True, aqe_k=3, rerank=True),
]
EVAL_RESULTS_PATH.write_text(json.dumps(eval_rows, indent=2), encoding="utf-8")
best_concat = max([row for row in eval_rows if row["concat_patch"]], key=lambda row: row["mAP"])
if best_concat["mAP"] >= 0.9154 and best_concat["R1"] >= 0.9833:
    verdict = "WIN"
elif best_concat["mAP"] >= 0.905 or best_concat["R1"] >= 0.980:
    verdict = "MARGINAL"
else:
    verdict = "FAIL"
summary = {
    "experiment": "14r_recovery_stage2",
    "verdict": verdict,
    "best_concat_row": best_concat,
    "outputs": {
        "source_prompts": str(PROMPTS_PATH),
        "best_mAP": str(BEST_MAP_PATH),
        "best_R1": str(BEST_R1_PATH),
        "last": str(LAST_PATH),
        "train_log": str(TRAIN_LOG_PATH),
        "eval_results": str(EVAL_RESULTS_PATH),
        "recipe": str(RECIPE_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))